<a href="https://colab.research.google.com/github/mohamedelguendy/Secure-Banking-System-Simulation/blob/main/Secure_Banking_System_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import re
import getpass
import json
import os
from datetime import datetime

# =========================
# FILES
# =========================
USERS_FILE = "/content/drive/MyDrive/Secure Banking System Simulation logs/users.json"
TRANSACTION_FILE = "/content/drive/MyDrive/Secure Banking System Simulation logs/transactions.json"
# =========================
# LOAD USERS (PERSISTENT)
# =========================
if os.path.exists(USERS_FILE):
    with open(USERS_FILE, "r") as f:
        users = json.load(f)
else:
    users = {}

# =========================
# LOAD TRANSACTIONS (PERSISTENT)
# =========================
if os.path.exists(TRANSACTION_FILE):
    with open(TRANSACTION_FILE, "r") as f:
        global_logs = json.load(f)
else:
    global_logs = []

current_user = None
transaction_counter = 1
MAX_ATTEMPTS = 5


# =========================
# SAVE FUNCTIONS
# =========================
def save_users():
    with open(USERS_FILE, "w") as f:
        json.dump(users, f, indent=4)


def save_transactions():
    with open(TRANSACTION_FILE, "w") as f:
        json.dump(global_logs, f, indent=4)


# =========================
# PASSWORD CHECKER
# =========================
def is_strong_password(password):
    return (
        len(password) >= 6 and
        re.search(r"[A-Z]", password) and
        re.search(r"[0-9]", password)
    )


# =========================
# TRANSACTION CREATOR
# =========================
def create_transaction(tx_type, sender, receiver, amount):
    global transaction_counter

    txn = {
        "id": f"TXN{transaction_counter:04d}",
        "type": tx_type,
        "from": sender,
        "to": receiver,
        "amount": amount,
        "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    transaction_counter += 1
    return txn


# =========================
# REGISTER
# =========================
def register():
    print("\n=== REGISTER USER ===")

    while True:
        username = input("Username (or exit): ")

        if username.lower() == "exit":
            return

        if username in users:
            print("User already exists!")
            continue

        password = getpass.getpass("Password: ")

        if not is_strong_password(password):
            print("Weak password!")
            print("- Min 6 chars")
            print("- 1 uppercase letter")
            print("- 1 number")
            continue

        users[username] = {
            "password": password,
            "balance": 0,
            "history": [],
            "failed_attempts": 0,
            "locked": False
        }

        save_users()
        print("User created successfully!")
        break


# =========================
# LOGIN
# =========================
def login():
    global current_user

    print("\n=== LOGIN ===")
    attempts_left = MAX_ATTEMPTS

    while attempts_left > 0:

        username = input("Username (or exit): ")

        if username.lower() == "exit":
            return False

        password = getpass.getpass("Password: ")

        if username not in users:
            print("User not found.")
            attempts_left -= 1
            print("Attempts left:", attempts_left)
            continue

        user = users[username]

        if user["locked"]:
            print("Account locked.")
            return False

        if password == user["password"]:
            print("Login successful!")
            user["failed_attempts"] = 0
            current_user = username
            save_users()
            return True

        attempts_left -= 1
        user["failed_attempts"] += 1

        print("Wrong password!")
        print("Attempts left:", attempts_left)

        if user["failed_attempts"] >= MAX_ATTEMPTS:
            user["locked"] = True
            save_users()
            print("Account locked!")
            return False

    return False


# =========================
# DEPOSIT
# =========================
def deposit():
    try:
        amount = float(input("Deposit amount: "))
    except:
        print("Invalid input!")
        return

    if amount <= 0:
        print("Invalid amount.")
        return

    users[current_user]["balance"] += amount

    txn = create_transaction("DEPOSIT", current_user, current_user, amount)

    users[current_user]["history"].append(txn)
    global_logs.append(txn)

    save_users()
    save_transactions()

    print("Deposit successful!")


# =========================
# TRANSFER
# =========================
def transfer():
    receiver = input("Receiver: ")

    try:
        amount = float(input("Amount: "))
    except:
        print("Invalid input!")
        return

    if receiver not in users:
        print("Receiver not found.")
        return

    if amount <= 0:
        print("Invalid amount.")
        return

    if users[current_user]["balance"] < amount:
        print("Insufficient balance.")
        return

    users[current_user]["balance"] -= amount
    users[receiver]["balance"] += amount

    txn = create_transaction("TRANSFER", current_user, receiver, amount)

    users[current_user]["history"].append(txn)
    users[receiver]["history"].append(txn)
    global_logs.append(txn)

    save_users()
    save_transactions()

    print("Transfer successful!")


# =========================
# VIEW HISTORY
# =========================
def view_history():
    print("\n=== TRANSACTION HISTORY ===")

    history = users[current_user]["history"]

    if not history:
        print("No transactions.")
        return

    for h in history:
        print(f"[{h['id']}] {h['type']} | From: {h['from']} | To: {h['to']} | Amount: {h['amount']} | Time: {h['time']}")


# =========================
# CHANGE PASSWORD
# =========================
def change_password():
    print("\n=== CHANGE PASSWORD ===")

    old = getpass.getpass("Old password: ")

    if old != users[current_user]["password"]:
        print("Incorrect password.")
        return

    while True:
        new = getpass.getpass("New password: ")

        if not is_strong_password(new):
            print("Weak password!")
            continue

        confirm = getpass.getpass("Confirm password: ")

        if new != confirm:
            print("Mismatch!")
            continue

        users[current_user]["password"] = new
        save_users()
        print("Password changed successfully!")
        break


# =========================
# USER MENU
# =========================
def user_menu():
    while True:
        print("\n=== USER MENU ===")
        print("1. Balance")
        print("2. Deposit")
        print("3. Transfer")
        print("4. History")
        print("5. Change Password")
        print("6. Logout")

        choice = input("Choose: ")

        if choice == "1":
            print("Balance:", users[current_user]["balance"])
        elif choice == "2":
            deposit()
        elif choice == "3":
            transfer()
        elif choice == "4":
            view_history()
        elif choice == "5":
            change_password()
        elif choice == "6":
            break
        else:
            print("Invalid option!")


# =========================
# MAIN SYSTEM
# =========================
def main():
    while True:
        print("\n=== BANK SYSTEM ===")
        print("1. Register")
        print("2. Login")
        print("3. View Global Logs")
        print("4. Exit")

        choice = input("Choose: ")

        if choice == "1":
            register()

        elif choice == "2":
            if login():
                user_menu()

        elif choice == "3":
            print("\n=== GLOBAL LOGS ===")
            for log in global_logs:
                print(f"[{log['id']}] {log['type']} | {log['from']} → {log['to']} | {log['amount']} | {log['time']}")

        elif choice == "4":
            print("Exiting system...")
            break

        else:
            print("Invalid choice!")


print("=== SIMPLE BANKING SYSTEM ===")
main()

=== SIMPLE BANKING SYSTEM ===

=== BANK SYSTEM ===
1. Register
2. Login
3. View Global Logs
4. Exit
Choose: 2

=== LOGIN ===
Username (or exit): mo
Password: ··········
Login successful!

=== USER MENU ===
1. Balance
2. Deposit
3. Transfer
4. History
5. Change Password
6. Logout
Choose: 4

=== TRANSACTION HISTORY ===
[TXN0001] DEPOSIT | From: mo | To: mo | Amount: 100000.0 | Time: 2026-05-08 14:30:53
[TXN0003] TRANSFER | From: jo | To: mo | Amount: 200.0 | Time: 2026-05-08 14:31:58

=== USER MENU ===
1. Balance
2. Deposit
3. Transfer
4. History
5. Change Password
6. Logout
Choose: 1
Balance: 100200.0

=== USER MENU ===
1. Balance
2. Deposit
3. Transfer
4. History
5. Change Password
6. Logout
